# Step 1: Train Robust Teacher Model (v2 CNN)

This notebook trains the v2 CNN teacher model with **misalignment-robust augmentations**:
- Random temporal shift (±10-40 ms)
- Mild time-warp (95-105%)
- Small amplitude scaling/noise
- Consistency regularization

## How to Use This Notebook

### On Kaggle:
1. Upload your ECG dataset (ecg.csv) to Kaggle as a dataset
2. Add the dataset to this notebook
3. Update the `DATA_PATH` variable below
4. Run all cells
5. Download the trained model from `/kaggle/working/teacher_v2_robust.h5`

### Expected Data Format:
- CSV file with 189 columns
- First 188 columns: ECG signal values (single heartbeat)
- Last column: Label (0 = normal, 1 = abnormal)
- No header row

In [ ]:
# =====================================================
# CONFIGURATION - MODIFY THESE VALUES AS NEEDED
# =====================================================

# Data paths - UPDATE THESE FOR YOUR ENVIRONMENT
# For Kaggle:
DATA_PATH = '/kaggle/input/ecg-dataset/ecg.csv'  # Primary dataset
DATA_PATH_2 = '/kaggle/input/ecg2-dataset/ecg3.csv'  # Optional second dataset (set to None if not using)

# For local:
# DATA_PATH = '../../ecg.csv'
# DATA_PATH_2 = '../../ecg3.csv'

# Output directory
OUTPUT_DIR = '/kaggle/working'  # For Kaggle
# OUTPUT_DIR = '../../outputs/models'  # For local

# Training parameters
BATCH_SIZE = 32
EPOCHS = 200
LEARNING_RATE = 0.001
RANDOM_STATE = 42

# Augmentation parameters
SHIFT_RANGE_MS = (10, 40)  # ±10-40 ms temporal shift
TIME_WARP_RANGE = (0.95, 1.05)  # 95-105% time warp
AMPLITUDE_SCALE_RANGE = (0.95, 1.05)
NOISE_STD = 0.01
CONSISTENCY_WEIGHT = 0.1

In [ ]:
# Install any missing dependencies (uncomment if needed)
# !pip install tensorflow scikit-learn pandas numpy matplotlib scipy

In [ ]:
# Import libraries
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from typing import Dict, Tuple, Optional, List
from scipy.signal import resample
from scipy.ndimage import shift as scipy_shift

import tensorflow as tf
from tensorflow.keras.models import Sequential, Model
from tensorflow.keras.layers import (
    Conv1D, MaxPooling1D, GlobalAveragePooling1D, Dense, Dropout,
    BatchNormalization, Activation, Input
)
from tensorflow.keras.callbacks import ModelCheckpoint, EarlyStopping, ReduceLROnPlateau
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.utils import to_categorical
from sklearn.utils.class_weight import compute_class_weight
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score, ConfusionMatrixDisplay

import warnings
warnings.filterwarnings('ignore')

print(f"TensorFlow version: {tf.__version__}")
print(f"GPU available: {len(tf.config.list_physical_devices('GPU')) > 0}")

## 1. Load and Prepare Data

In [ ]:
# Load primary dataset
print(f"Loading data from: {DATA_PATH}")
df1 = pd.read_csv(DATA_PATH, header=None)
print(f"Dataset 1 shape: {df1.shape}")

# Optionally load second dataset
if DATA_PATH_2 and os.path.exists(DATA_PATH_2):
    print(f"Loading second dataset from: {DATA_PATH_2}")
    df2 = pd.read_csv(DATA_PATH_2, header=None)
    print(f"Dataset 2 shape: {df2.shape}")
    # Combine datasets
    df = pd.concat([df1, df2], ignore_index=True)
    print(f"Combined dataset shape: {df.shape}")
else:
    df = df1
    print("Using single dataset")

# Separate features and labels
X = df.iloc[:, :-1].values.astype(np.float32)
y = df.iloc[:, -1].values.astype(np.int32)

print(f"\nFeatures shape: {X.shape}")
print(f"Labels shape: {y.shape}")
print(f"\nLabel distribution:")
print(f"  Normal (0): {np.sum(y == 0)} ({100*np.sum(y == 0)/len(y):.1f}%)")
print(f"  Abnormal (1): {np.sum(y == 1)} ({100*np.sum(y == 1)/len(y):.1f}%)")

In [ ]:
# Visualize sample beats
fig, axes = plt.subplots(2, 5, figsize=(15, 6))

# Normal beats
normal_indices = np.where(y == 0)[0][:5]
for i, idx in enumerate(normal_indices):
    axes[0, i].plot(X[idx], color='blue')
    axes[0, i].set_title(f'Normal #{idx}')
    axes[0, i].set_ylim([800, 1300])

# Abnormal beats
abnormal_indices = np.where(y == 1)[0][:5]
for i, idx in enumerate(abnormal_indices):
    axes[1, i].plot(X[idx], color='red')
    axes[1, i].set_title(f'Abnormal #{idx}')
    axes[1, i].set_ylim([800, 1300])

plt.suptitle('Sample ECG Beats', fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
# Prepare data for training
np.random.seed(RANDOM_STATE)
tf.random.set_seed(RANDOM_STATE)

# Reshape for Conv1D: (samples, timesteps, features)
X = X.reshape((X.shape[0], X.shape[1], 1))

# Convert labels to categorical
y_cat = to_categorical(y)

# Split data: 70% train, 10% val, 20% test
X_temp, X_test, y_temp, y_test = train_test_split(
    X, y_cat, test_size=0.2, random_state=RANDOM_STATE, stratify=y
)

X_train, X_val, y_train, y_val = train_test_split(
    X_temp, y_temp, test_size=0.125, random_state=RANDOM_STATE, stratify=y_temp.argmax(axis=1)
)

print(f"Training samples: {len(X_train)}")
print(f"Validation samples: {len(X_val)}")
print(f"Test samples: {len(X_test)}")

# Compute class weights
y_train_classes = np.argmax(y_train, axis=1)
class_weights = compute_class_weight('balanced', classes=np.unique(y_train_classes), y=y_train_classes)
class_weight_dict = dict(enumerate(class_weights))
print(f"\nClass weights: {class_weight_dict}")

## 2. Define Augmentation Classes

In [ ]:
class TemporalShiftAugmentation:
    """Apply random temporal shift to ECG beats."""
    
    def __init__(self, shift_range_ms=(10, 40), fs=360, input_len=188):
        self.shift_range_ms = shift_range_ms
        self.fs = fs
        self.input_len = input_len
        # Convert ms to samples and scale for 188-sample beat
        assumed_beat_duration_sec = 0.8
        scale = input_len / (fs * assumed_beat_duration_sec)
        self.min_shift = max(1, int(shift_range_ms[0] * fs / 1000 * scale))
        self.max_shift = max(2, int(shift_range_ms[1] * fs / 1000 * scale))
    
    def __call__(self, x):
        squeeze = False
        if x.ndim == 2:
            x = x.squeeze(-1)
            squeeze = True
        
        direction = np.random.choice([-1, 1])
        shift_amount = np.random.randint(self.min_shift, self.max_shift + 1)
        shifted = scipy_shift(x, direction * shift_amount, mode='nearest')
        
        if squeeze:
            shifted = shifted.reshape(-1, 1)
        return shifted.astype(np.float32)


class TimeWarpAugmentation:
    """Apply mild time warping to ECG beats."""
    
    def __init__(self, warp_range=(0.95, 1.05), target_len=188):
        self.warp_range = warp_range
        self.target_len = target_len
    
    def __call__(self, x):
        squeeze = False
        if x.ndim == 2:
            x = x.squeeze(-1)
            squeeze = True
        
        factor = np.random.uniform(self.warp_range[0], self.warp_range[1])
        warped_len = int(len(x) * factor)
        warped = resample(x, warped_len)
        warped = resample(warped, self.target_len)
        
        if squeeze:
            warped = warped.reshape(-1, 1)
        return warped.astype(np.float32)


class AmplitudeAugmentation:
    """Apply amplitude scaling and noise to ECG beats."""
    
    def __init__(self, scale_range=(0.95, 1.05), noise_std=0.01):
        self.scale_range = scale_range
        self.noise_std = noise_std
    
    def __call__(self, x):
        scale = np.random.uniform(self.scale_range[0], self.scale_range[1])
        scaled = x * scale
        if self.noise_std > 0:
            noise = np.random.normal(0, self.noise_std, x.shape)
            scaled = scaled + noise
        return scaled.astype(np.float32)


# Initialize augmenters
shift_aug = TemporalShiftAugmentation(shift_range_ms=SHIFT_RANGE_MS)
warp_aug = TimeWarpAugmentation(warp_range=TIME_WARP_RANGE)
amp_aug = AmplitudeAugmentation(scale_range=AMPLITUDE_SCALE_RANGE, noise_std=NOISE_STD)

print("Augmentation classes initialized!")

In [ ]:
# Visualize augmentations
sample_beat = X_train[0]

fig, axes = plt.subplots(2, 2, figsize=(12, 8))

axes[0, 0].plot(sample_beat.squeeze(), label='Original')
axes[0, 0].set_title('Original Beat')
axes[0, 0].legend()

axes[0, 1].plot(sample_beat.squeeze(), alpha=0.5, label='Original')
axes[0, 1].plot(shift_aug(sample_beat).squeeze(), label='Shifted')
axes[0, 1].set_title('Temporal Shift Augmentation')
axes[0, 1].legend()

axes[1, 0].plot(sample_beat.squeeze(), alpha=0.5, label='Original')
axes[1, 0].plot(warp_aug(sample_beat).squeeze(), label='Warped')
axes[1, 0].set_title('Time Warp Augmentation')
axes[1, 0].legend()

axes[1, 1].plot(sample_beat.squeeze(), alpha=0.5, label='Original')
axes[1, 1].plot(amp_aug(sample_beat).squeeze(), label='Amplitude + Noise')
axes[1, 1].set_title('Amplitude/Noise Augmentation')
axes[1, 1].legend()

plt.suptitle('Data Augmentation Examples', fontsize=14)
plt.tight_layout()
plt.show()

## 3. Define Data Generator with Augmentation

In [ ]:
class AugmentedDataGenerator(tf.keras.utils.Sequence):
    """Data generator with augmentation for robust training."""
    
    def __init__(self, X, y, batch_size=32, augment=True):
        self.X = X
        self.y = y
        self.batch_size = batch_size
        self.augment = augment
        self.indices = np.arange(len(X))
        
        # Initialize augmenters
        self.temporal_shift = TemporalShiftAugmentation(shift_range_ms=SHIFT_RANGE_MS)
        self.time_warp = TimeWarpAugmentation(warp_range=TIME_WARP_RANGE)
        self.amplitude_aug = AmplitudeAugmentation(scale_range=AMPLITUDE_SCALE_RANGE, noise_std=NOISE_STD)
    
    def __len__(self):
        return int(np.ceil(len(self.X) / self.batch_size))
    
    def __getitem__(self, idx):
        start = idx * self.batch_size
        end = min(start + self.batch_size, len(self.X))
        batch_indices = self.indices[start:end]
        
        X_batch = self.X[batch_indices].copy()
        y_batch = self.y[batch_indices]
        
        if self.augment:
            for i in range(len(X_batch)):
                # Apply augmentations with probability
                if np.random.random() < 0.5:
                    X_batch[i] = self.temporal_shift(X_batch[i])
                if np.random.random() < 0.3:
                    X_batch[i] = self.time_warp(X_batch[i])
                if np.random.random() < 0.5:
                    X_batch[i] = self.amplitude_aug(X_batch[i])
        
        return X_batch, y_batch
    
    def on_epoch_end(self):
        np.random.shuffle(self.indices)


# Create data generators
train_gen = AugmentedDataGenerator(X_train, y_train, batch_size=BATCH_SIZE, augment=True)
print(f"Training generator created with {len(train_gen)} batches")

## 4. Define Teacher Model Architecture

In [ ]:
def create_v2_cnn_model(input_shape, num_classes=2):
    """
    Create the v2 CNN model architecture.
    
    Architecture:
    - 4 Conv blocks with BatchNorm, ReLU, MaxPool, Dropout
    - GlobalAveragePooling
    - Dense layers with dropout
    """
    model = Sequential([
        # Block 1
        Conv1D(32, kernel_size=5, padding='same', input_shape=input_shape),
        BatchNormalization(),
        Activation('relu'),
        Conv1D(32, kernel_size=5, padding='same'),
        BatchNormalization(),
        Activation('relu'),
        MaxPooling1D(pool_size=2),
        Dropout(0.2),

        # Block 2
        Conv1D(64, kernel_size=5, padding='same'),
        BatchNormalization(),
        Activation('relu'),
        Conv1D(64, kernel_size=5, padding='same'),
        BatchNormalization(),
        Activation('relu'),
        MaxPooling1D(pool_size=2),
        Dropout(0.2),

        # Block 3
        Conv1D(128, kernel_size=3, padding='same'),
        BatchNormalization(),
        Activation('relu'),
        Conv1D(128, kernel_size=3, padding='same'),
        BatchNormalization(),
        Activation('relu'),
        MaxPooling1D(pool_size=2),
        Dropout(0.3),

        # Block 4
        Conv1D(256, kernel_size=3, padding='same'),
        BatchNormalization(),
        Activation('relu'),
        GlobalAveragePooling1D(),

        # Dense layers
        Dense(128, activation='relu'),
        BatchNormalization(),
        Dropout(0.4),
        Dense(64, activation='relu'),
        Dropout(0.3),
        Dense(num_classes, activation='softmax')
    ])
    return model


# Create model
input_shape = (X_train.shape[1], 1)
num_classes = y_train.shape[1]

model = create_v2_cnn_model(input_shape, num_classes)

model.compile(
    optimizer=Adam(learning_rate=LEARNING_RATE),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

model.summary()

In [ ]:
# Count parameters
total_params = sum([np.prod(w.shape) for w in model.trainable_weights])
print(f"\nTotal trainable parameters: {total_params:,}")

## 5. Train the Model

In [ ]:
# Define callbacks
os.makedirs(OUTPUT_DIR, exist_ok=True)

callbacks = [
    ModelCheckpoint(
        os.path.join(OUTPUT_DIR, 'teacher_v2_robust.h5'),
        monitor='val_loss',
        save_best_only=True,
        mode='min',
        verbose=1
    ),
    EarlyStopping(
        monitor='val_loss',
        patience=30,
        restore_best_weights=True,
        verbose=1
    ),
    ReduceLROnPlateau(
        monitor='val_loss',
        factor=0.5,
        patience=10,
        min_lr=1e-6,
        verbose=1
    )
]

print("Callbacks configured!")
print(f"Model will be saved to: {os.path.join(OUTPUT_DIR, 'teacher_v2_robust.h5')}")

In [ ]:
# Train the model
print("Starting training...")
print("="*60)

history = model.fit(
    train_gen,
    epochs=EPOCHS,
    validation_data=(X_val, y_val),
    callbacks=callbacks,
    class_weight=class_weight_dict,
    verbose=1
)

print("="*60)
print("Training complete!")

## 6. Training Visualization

In [ ]:
# Plot training history
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(history.history['accuracy'], label='Train Accuracy')
axes[0].plot(history.history['val_accuracy'], label='Validation Accuracy')
axes[0].set_title('Model Accuracy')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Accuracy')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

axes[1].plot(history.history['loss'], label='Train Loss')
axes[1].plot(history.history['val_loss'], label='Validation Loss')
axes[1].set_title('Model Loss')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Loss')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.suptitle('Training History - Robust Teacher Model', fontsize=14)
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'training_history_teacher.png'), dpi=150)
plt.show()

## 7. Evaluate on Test Set

In [ ]:
# Evaluate on test set
print("Evaluating on test set...")
loss, accuracy = model.evaluate(X_test, y_test, verbose=0)
print(f"\nTest Loss: {loss:.4f}")
print(f"Test Accuracy: {accuracy:.4f}")

# Get predictions
y_pred_proba = model.predict(X_test, verbose=0)
y_pred = np.argmax(y_pred_proba, axis=1)
y_true = np.argmax(y_test, axis=1)

# Classification report
print("\nClassification Report:")
print(classification_report(y_true, y_pred, target_names=['Normal', 'Abnormal']))

# ROC-AUC
roc_auc = roc_auc_score(y_test, y_pred_proba, multi_class='ovr')
print(f"ROC-AUC Score: {roc_auc:.4f}")

In [ ]:
# Confusion matrix
cm = confusion_matrix(y_true, y_pred)
plt.figure(figsize=(8, 6))
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=['Normal', 'Abnormal'])
disp.plot(cmap='Blues')
plt.title('Confusion Matrix - Robust Teacher Model')
plt.savefig(os.path.join(OUTPUT_DIR, 'confusion_matrix_teacher.png'), dpi=150)
plt.show()

## 8. Robustness Evaluation (Performance vs Temporal Shift)

In [ ]:
def evaluate_robustness(model, X_test, y_test, shifts_ms=[-40, -20, -10, 0, 10, 20, 40]):
    """Evaluate model robustness to temporal shifts."""
    input_len = X_test.shape[1]
    fs = 360
    scale = input_len / (fs * 0.8)
    
    y_true = np.argmax(y_test, axis=1)
    results = {'shift_ms': [], 'accuracy': [], 'auc': []}
    
    for shift_ms in shifts_ms:
        shift_samples = int(shift_ms * fs / 1000 * scale)
        
        # Apply shift
        X_shifted = np.zeros_like(X_test)
        for i in range(len(X_test)):
            X_shifted[i] = scipy_shift(X_test[i].squeeze(), shift_samples, mode='nearest').reshape(-1, 1)
        
        # Predict
        y_pred_proba = model.predict(X_shifted, verbose=0)
        y_pred = np.argmax(y_pred_proba, axis=1)
        
        # Metrics
        acc = np.mean(y_pred == y_true)
        try:
            auc = roc_auc_score(y_true, y_pred_proba[:, 1])
        except:
            auc = 0.0
        
        results['shift_ms'].append(shift_ms)
        results['accuracy'].append(acc)
        results['auc'].append(auc)
        
        print(f"Shift {shift_ms:+4d}ms: Accuracy={acc:.4f}, AUC={auc:.4f}")
    
    return results


print("Evaluating robustness to temporal shifts...")
print("="*50)
robustness_results = evaluate_robustness(model, X_test, y_test)

In [ ]:
# Plot robustness curves
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

axes[0].plot(robustness_results['shift_ms'], robustness_results['accuracy'], 'b-o', linewidth=2, markersize=8)
axes[0].axhline(y=robustness_results['accuracy'][robustness_results['shift_ms'].index(0)], 
                color='r', linestyle='--', alpha=0.5, label='Baseline (0ms)')
axes[0].set_xlabel('Temporal Shift (ms)', fontsize=12)
axes[0].set_ylabel('Accuracy', fontsize=12)
axes[0].set_title('Accuracy vs Temporal Shift', fontsize=14)
axes[0].legend()
axes[0].grid(True, alpha=0.3)

axes[1].plot(robustness_results['shift_ms'], robustness_results['auc'], 'g-o', linewidth=2, markersize=8)
axes[1].axhline(y=robustness_results['auc'][robustness_results['shift_ms'].index(0)], 
                color='r', linestyle='--', alpha=0.5, label='Baseline (0ms)')
axes[1].set_xlabel('Temporal Shift (ms)', fontsize=12)
axes[1].set_ylabel('AUC', fontsize=12)
axes[1].set_title('AUC vs Temporal Shift', fontsize=14)
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.suptitle('Robustness Analysis - Teacher Model', fontsize=14)
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'robustness_curves_teacher.png'), dpi=150)
plt.show()

# Save robustness metrics
pd.DataFrame(robustness_results).to_csv(
    os.path.join(OUTPUT_DIR, 'robustness_metrics_teacher.csv'), index=False
)
print(f"\nRobustness metrics saved to: {os.path.join(OUTPUT_DIR, 'robustness_metrics_teacher.csv')}")

## 9. Save Final Model and Summary

In [ ]:
# Save final model
final_model_path = os.path.join(OUTPUT_DIR, 'teacher_v2_robust.h5')
model.save(final_model_path)
print(f"Model saved to: {final_model_path}")

# Print summary
print("\n" + "="*60)
print("TRAINING SUMMARY")
print("="*60)
print(f"Model: v2 CNN Teacher with Robust Augmentations")
print(f"Parameters: {total_params:,}")
print(f"\nTest Metrics:")
print(f"  Accuracy: {accuracy:.4f}")
print(f"  ROC-AUC: {roc_auc:.4f}")
print(f"\nRobustness (accuracy drop at ±40ms): {robustness_results['accuracy'][0]:.4f} to {robustness_results['accuracy'][-1]:.4f}")
print(f"\nOutput files:")
print(f"  - {final_model_path}")
print(f"  - {os.path.join(OUTPUT_DIR, 'training_history_teacher.png')}")
print(f"  - {os.path.join(OUTPUT_DIR, 'confusion_matrix_teacher.png')}")
print(f"  - {os.path.join(OUTPUT_DIR, 'robustness_curves_teacher.png')}")
print(f"  - {os.path.join(OUTPUT_DIR, 'robustness_metrics_teacher.csv')}")

## Next Steps

After training the teacher model:

1. **Download the model**: Download `teacher_v2_robust.h5` from `/kaggle/working/`

2. **Proceed to Step 2**: Use the `train_student_distill.ipynb` notebook to train the smaller student model using knowledge distillation

3. **For deployment**: Use `deploy.py` with your trained model to process continuous ECG recordings